# 沪深300成分股数据
## 1. 获取权重数据

用 `pd.concat()` 把列表里的多张表**上下拼成一张**。

> **`pd.concat()`**：Pandas 中用来沿某个方向（行或列）拼接多个 DataFrame / Series 的主要函数。

- `axis=0`：上下拼（默认）
- `axis=1`：左右拼

| 函数 | 作用 |
|---|---|
| `unique()` | 返回去重后的值 |
| `nunique()` | 返回去重后的个数 |

In [1]:
import tushare as ts
import pandas as pd

In [2]:
pro = ts.pro_api()  # 给连接tushare的接口起名为pro

frames = [] #list什么都能装，df也能装

for year in range(2019, 2025): #年份是2019-2024，不含2025
    df = pro.index_weight(index_code='000300.SH', #pro.index_weight() tushare的指令，返回指数在指定时间段内的成分股和权重
                          start_date=f'{year}0101',
                          end_date=f'{year}1231') #时间跨度是从20190101到20241231
    frames.append(df) #把这一年的数据加到列表末尾

    print(f'==================== {year} ====================')
    df.info()  # 直接调用，不需要套 print
    print(df.head())
    print('frames 里目前有', len(frames), '张表')


weights = pd.concat(frames, ignore_index=True) #把列表里的六张表上下拼成一张，并且重新编号
weights.to_csv('hs300_weights.csv', index=False)

print('===================================================================')

print('曾进入指数的股票数：', weights['con_code'].nunique())
print('===================================================================')

print('调取了多少个横截面（日期）：', weights['trade_date'].nunique())          # 一共有几个快照日期，12*6 = 72个
print('===================================================================')

print('前13个日期是：', weights['trade_date'].sort_values().unique()[:13])  # 看前 13 个日期长什么样
print('===================================================================')

print('日期的详情：', weights.groupby('trade_date').size().describe())     # 一共多少日期，每个日期都有多少entries等
print('===================================================================')

weights.info() #是dataframe 21600 = 300*12*6行


==================== 2019 ====================
<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   index_code  3600 non-null   str    
 1   con_code    3600 non-null   str    
 2   trade_date  3600 non-null   str    
 3   weight      3600 non-null   float64
dtypes: float64(1), str(3)
memory usage: 204.0 KB
  index_code   con_code trade_date  weight
0  000300.SH  601318.SH   20191231   6.715
1  000300.SH  600519.SH   20191231   4.312
2  000300.SH  600036.SH   20191231   2.812
3  000300.SH  000651.SZ   20191231   2.289
4  000300.SH  601166.SH   20191231   2.088
frames 里目前有 1 张表
==================== 2020 ====================
<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   index_code  3600 non-null   str    
 1   con_code    3600 

## 2. 拼接多年的日线数据

### 2.1 tushare `daily` 笔记

> 前提：`daily` 给的是**未复权**价格，且默认**日期倒序**（用前先 `sort_values('trade_date')`）。

#### 一、各列含义

| 列名 | 含义 | 单位 |
|---|---|---|
| `ts_code` | 股票代码 | — |
| `trade_date` | 交易日期（8 位字符串） | — |
| `open` | 开盘价（集合竞价产生） | 元 |
| `high` | 最高价 | 元 |
| `low` | 最低价 | 元 |
| `close` | 收盘价 | 元 |
| `pre_close` | 昨收价，除权除息日会调整 | 元 |
| `change` | 涨跌额 = close − pre_close | 元 |
| `pct_chg` | 涨跌幅 = change ÷ pre_close × 100 | **%**（用时 ÷100） |
| `vol` | 成交量 | **手**（1 手 = 100 股） |
| `amount` | 成交额 | **千元** |

**单位自检**：成交均价 = amount × 1000 ÷ (vol × 100)，应落在 low 和 high 之间。

#### 二、为什么用 `pre_close`，不直接用上一行的 `close`

- 普通交易日：`pre_close` = 前一天 `close`。
- **除权除息日**：交易所把昨收调低，扣掉分红/送股的部分。
- 复权价格英文里是 **Adjusted price**

例：昨收 10 元，每股派现 0.5 元，今天收 9.5 元。

| 算法 | 结果 | |
|---|---|---|
| 用前一天 close | (9.5 − 10) / 10 = −5% | ✗ 钱只是从股价变成现金，股东没亏 |
| 用 pre_close (9.5) | (9.5 − 9.5) / 9.5 = 0% | ✓ |

**结论**：
- `pct_chg` 基于调整后的 `pre_close`，是真实日收益率。
- 未复权 `close` 在除权日有假跳空，**不要用 `close.pct_change()` 算收益**；用 `pct_chg / 100` 或复权价。

### 2.2 拼接实操

In [3]:
import time

codes = weights['con_code'].unique() #unique() 给出这些值本身，nunique() 只给出个数
print(type(codes)) #返回的是 pandas/numpy 体系里的数组

test = pro.daily(ts_code=codes[0], start_date='20190101', end_date='20241231') #pro.daily(),每日交易数据的调用接口 #第一个参数是股票代码

print(test.shape)
test.head()

<class 'pandas.arrays.ArrowStringArray'>
(1456, 11)


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,601318.SH,20241231,53.79,54.02,52.65,52.65,53.90,-1.25,-2.3191,677117.90,3607846.474
1,601318.SH,20241230,53.22,53.90,53.21,53.90,53.27,0.63,1.1827,533437.83,2865180.504
2,601318.SH,20241227,53.45,53.75,52.71,53.27,53.45,-0.18,-0.3368,537757.19,2862446.917
3,601318.SH,20241226,53.89,53.94,53.36,53.45,53.90,-0.45,-0.8349,379221.70,2031175.519
4,601318.SH,20241225,53.81,54.31,53.57,53.90,53.78,0.12,0.2231,417398.96,2249897.990


In [4]:
daily_list, adj_list, failed = [], [], []

for i, code in enumerate(codes): #enumerate 会把列表变成一对一对的 (序号, 元素), i这里的主要作用是算进度
    try:
        daily_list.append(pro.daily(ts_code=code, start_date='20190101', end_date='20241231'))
        adj_list.append(pro.adj_factor(ts_code=code, start_date='20190101', end_date='20241231')) #pro.adj_factor: Tushare 的复权因子接口
    except Exception as e: #先尝试执行 try 里的代码；如果出错，不让程序停下，而是执行 except 里的代码。
        failed.append(code) #as e: 把这次具体的错误对象存进变量 e，方便后面打印或分析
        print(code, '失败:', e) #e 是错误信息，，可以换成任何合法的变量名比如err
    time.sleep(0.3) #暂停 0.3 秒，避免调用太快超过接口的每分钟次数限制
    if i % 50 == 0:
        print(f'进度 {i}/{len(codes)}')

daily = pd.concat(daily_list)
adj = pd.concat(adj_list)
daily.to_csv('hs300_daily.csv', index=False)
adj.to_csv('hs300_adj.csv', index=False)

print('日线行数:', len(daily))
print('失败的股票:', failed)

进度 0/499
进度 50/499
进度 100/499
进度 150/499
进度 200/499
进度 250/499
进度 300/499
进度 350/499
进度 400/499
进度 450/499
日线行数: 693861
失败的股票: []


In [5]:
daily.head()

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,601318.SH,20241231,53.79,54.02,52.65,52.65,53.90,-1.25,-2.3191,677117.90,3607846.474
1,601318.SH,20241230,53.22,53.90,53.21,53.90,53.27,0.63,1.1827,533437.83,2865180.504
2,601318.SH,20241227,53.45,53.75,52.71,53.27,53.45,-0.18,-0.3368,537757.19,2862446.917
3,601318.SH,20241226,53.89,53.94,53.36,53.45,53.90,-0.45,-0.8349,379221.70,2031175.519
4,601318.SH,20241225,53.81,54.31,53.57,53.90,53.78,0.12,0.2231,417398.96,2249897.990


In [6]:
daily = daily.reset_index(drop=True)

### 2.3 检查每个股票交易日Coverage

In [7]:
daily['trade_date'] = pd.to_datetime(daily['trade_date']) #把证书或者字符变成真正的日期类型，可以用于比较、按年月划分

coverage = daily.groupby('ts_code')['trade_date'].agg(['min', 'max', 'count']) #对每组同时算三个统计量——最早日期、最晚日期、行数（即交易日数）。
coverage.columns = ['首个交易日', '最后交易日', '交易日数']
coverage = coverage.sort_values('交易日数')
coverage.head(20)

,首个交易日,最后交易日,交易日数
ts_code,,,
002450.SZ,2019-01-02,2021-05-28,151
603296.SH,2023-08-08,2024-12-31,340
688472.SH,2023-06-09,2024-12-31,380
601059.SH,2023-02-01,2024-12-31,468
688506.SH,2023-01-06,2024-12-31,481
601136.SH,2022-12-22,2024-12-31,491
688271.SH,2022-08-22,2024-12-31,573
688041.SH,2022-08-12,2024-12-31,579
301269.SZ,2022-07-29,2024-12-31,589


In [8]:
coverage.tail(20)

,首个交易日,最后交易日,交易日数
ts_code,,,
600029.SH,2019-01-02,2024-12-31,1456
600028.SH,2019-01-02,2024-12-31,1456
600025.SH,2019-01-02,2024-12-31,1456
600023.SH,2019-01-02,2024-12-31,1456
600019.SH,2019-01-02,2024-12-31,1456
600018.SH,2019-01-02,2024-12-31,1456
600016.SH,2019-01-02,2024-12-31,1456
600015.SH,2019-01-02,2024-12-31,1456
600011.SH,2019-01-02,2024-12-31,1456


# 跑通后的三个练习

**每个都只需要改一两处，目的是确认你真的能改。**

1. 换一个指数：把 '000300.SH' 改成中证 500 的 '000905.SH'，重新跑第三步，看曾经入选的股票有多少只。改完记得改回来，或者存成不同的文件名，别覆盖了沪深 300 的文件。
2. 按年统计：在 weights 里加一列年份，算出每一年有多少只不同的股票出现过。提示：trade_date 的前四位就是年份。
3. 找出被调出的股票：找出 2019 年 1 月在指数里、但 2024 年 12 月已经不在的股票，看看有多少只。这些就是"如果用今天的成分股回测，会被漏掉的股票"。

## 练习一：把 '000300.SH' 改成中证 500 的 '000905.SH'
结论：**曾经有998个股票入选，新的文件名 zz500_weights.csv**

In [9]:
frames_zz = [] 

for year in range(2019, 2025):
    df_zz = pro.index_weight(index_code='000905.SH', 
                          start_date=f'{year}0101',
                          end_date=f'{year}1231') 
    frames_zz.append(df_zz) 

    print(f'==================== {year} ====================')
    df_zz.info()  # 直接调用，不需要套 print
    print('frames_zz 里目前有', len(frames_zz), '张表')


weights_zz = pd.concat(frames_zz, ignore_index=True) 
weights_zz.to_csv('zz500_weights.csv', index=False)

print('===================================================================')

print('曾进入指数的股票数：', weights_zz['con_code'].nunique())
print('===================================================================')

print('调取了多少个横截面（日期）：', weights_zz['trade_date'].nunique())          
print('===================================================================')

print('前13个日期是：', weights_zz['trade_date'].sort_values().unique()[:13])  
print('===================================================================')

print('日期的详情：', weights_zz.groupby('trade_date').size().describe())     # 一共多少日期，每个日期都有多少entries等
print('===================================================================')

weights_zz.info() #500*12*6 = 36000，一共3.6万个entries，每年6000个

==================== 2019 ====================
<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   index_code  6000 non-null   str    
 1   con_code    6000 non-null   str    
 2   trade_date  6000 non-null   str    
 3   weight      6000 non-null   float64
dtypes: float64(1), str(3)
memory usage: 340.0 KB
frames_zz 里目前有 1 张表
==================== 2020 ====================
<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   index_code  6000 non-null   str    
 1   con_code    6000 non-null   str    
 2   trade_date  6000 non-null   str    
 3   weight      6000 non-null   float64
dtypes: float64(1), str(3)
memory usage: 340.0 KB
frames_zz 里目前有 2 张表
==================== 2021 ====================
<class 'pandas.DataFrame'>
RangeIndex

## 练习二：按年统计
- 在 weights 里加一列年份，算出每一年有多少只不同的股票出现过。提示：trade_date 的前四位就是年份。

In [10]:
weights.head()

,index_code,con_code,trade_date,weight
0,000300.SH,601318.SH,20191231,6.715
1,000300.SH,600519.SH,20191231,4.312
2,000300.SH,600036.SH,20191231,2.812
3,000300.SH,000651.SZ,20191231,2.289
4,000300.SH,601166.SH,20191231,2.088


In [11]:
weights['trade_year'] = weights['trade_date'].astype(str).str[:4]
weights.head()

,index_code,con_code,trade_date,weight,trade_year
0,000300.SH,601318.SH,20191231,6.715,2019
1,000300.SH,600519.SH,20191231,4.312,2019
2,000300.SH,600036.SH,20191231,2.812,2019
3,000300.SH,000651.SZ,20191231,2.289,2019
4,000300.SH,601166.SH,20191231,2.088,2019


- s[:4]       # 对 Series 切片：取前四行
- s.str[:4]   # 对 Series 中的每个字符串切片：每个元素取前四个字符

In [20]:
for year in range(2019,2025):
    stocks_per_year = weights[weights['trade_year'] == str(year)]
    print(f'{year}年出现的股票数: ', stocks_per_year['con_code'].nunique())

2019年出现的股票数:  335
2020年出现的股票数:  347
2021年出现的股票数:  353
2022年出现的股票数:  343
2023年出现的股票数:  323
2024年出现的股票数:  328


### 练习三 找出曾经在但是后面不在的股票
- 找出被调出的股票：找出 2019 年 1 月在指数里、但 2024 年 12 月已经不在的股票，看看有多少只。这些就是"如果用今天的成分股回测，会被漏掉的股票"。

In [14]:
weights['trade_month'] = weights['trade_date'].astype(str).str[:6]
weights.head()

,index_code,con_code,trade_date,weight,trade_year,trade_month
0,000300.SH,601318.SH,20191231,6.715,2019,201912
1,000300.SH,600519.SH,20191231,4.312,2019,201912
2,000300.SH,600036.SH,20191231,2.812,2019,201912
3,000300.SH,000651.SZ,20191231,2.289,2019,201912
4,000300.SH,601166.SH,20191231,2.088,2019,201912


In [15]:
stock_list_201901 = weights[weights['trade_month'] == '201901']['con_code'].unique()

In [16]:
stock_list_202412 = weights[weights['trade_month'] == '202412']['con_code'].unique()

In [17]:
print(stock_list_201901)
print(stock_list_202412)

<ArrowStringArray>
['601318.SH', '600519.SH', '600036.SH', '601166.SH', '000333.SZ', '000651.SZ',
 '601328.SH', '600016.SH', '600887.SH', '600030.SH',
 ...
 '002925.SZ', '002625.SZ', '600390.SH', '601828.SH', '603260.SH', '601838.SH',
 '601108.SH', '000553.SZ', '601212.SH', '603156.SH']
Length: 300, dtype: str
<ArrowStringArray>
['600519.SH', '300750.SZ', '601318.SH', '600036.SH', '000333.SZ', '600900.SH',
 '300059.SZ', '600030.SH', '601166.SH', '000858.SZ',
 ...
 '688187.SH', '601236.SH', '300979.SZ', '601136.SH', '000708.SZ', '601808.SH',
 '688082.SH', '000800.SZ', '600377.SH', '001289.SZ']
Length: 300, dtype: str


In [19]:
set_old = set(stock_list_201901)-set(stock_list_202412)
print(set_old)
print(len(set_old))

{'002558.SZ', '600038.SH', '300296.SZ', '300024.SZ', '601155.SH', '300136.SZ', '600498.SH', '600390.SH', '000826.SZ', '600340.SH', '603160.SH', '601991.SH', '601198.SH', '600398.SH', '600352.SH', '600998.SH', '600221.SH', '002294.SZ', '601216.SH', '600705.SH', '002450.SZ', '600208.SH', '600177.SH', '000783.SZ', '002625.SZ', '600516.SH', '000402.SZ', '600109.SH', '601555.SH', '000703.SZ', '002773.SZ', '600637.SH', '002797.SZ', '600566.SH', '000415.SZ', '600739.SH', '600704.SH', '601212.SH', '601108.SH', '300144.SZ', '300142.SZ', '002925.SZ', '300070.SZ', '600297.SH', '002202.SZ', '600606.SH', '002032.SZ', '002065.SZ', '600369.SH', '601828.SH', '600518.SH', '603858.SH', '600170.SH', '300003.SZ', '002008.SZ', '600100.SH', '600339.SH', '600977.SH', '600004.SH', '600703.SH', '002411.SZ', '000553.SZ', '601228.SH', '600383.SH', '600068.SH', '600583.SH', '000839.SZ', '000627.SZ', '002310.SZ', '002044.SZ', '000069.SZ', '601333.SH', '601018.SH', '603156.SH', '600522.SH', '002572.SZ', '002468.SZ'